# Avenue Dataset — Anomaly Detection

**Pipeline:** YOLOv8 object detection → rule-based labeling → ResNet-34 classifier

| Step | Description |
|------|-------------|
| 1 | Extract frames from `.avi` surveillance videos |
| 2 | Run YOLOv8 on each frame to detect objects |
| 3 | Assign rule-based anomaly labels using GT `.mat` files |
| 4 | Train ResNet-34 classifier on the labeled frames |
| 5 | Evaluate and export the model |

**Dataset:** CUHK Avenue — 16 training videos (normal only), 21 test videos (with anomalies), 640×360 @ ~25 FPS  
**Classes:** `normal` · `unusual action` · `abnormal object`

## 1  Setup

In [2]:
pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 MB 46.3 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:
      Successfully uninstalled torch-2.11.0

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Users/stuti_up_02/anaconda3/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 2  Configuration

In [2]:
DEVICE = torch.device(
    'mps'  if torch.backends.mps.is_available()  else
    'cuda' if torch.cuda.is_available()           else
    'cpu'
)
print(f'Device: {DEVICE}')

BASE_DIR   = Path('datasets/Avenue Dataset')
VIDEO_DIR  = BASE_DIR / 'testing_videos'
FRAME_DIR  = BASE_DIR / 'frames/test'
LABEL_DIR  = BASE_DIR / 'labels/test'
GT_MAT_DIR = BASE_DIR / 'testing_vol'
CSV_PATH   = Path('ave_anomaly_labels.csv')

LABEL2IDX = {'normal': 0, 'unusual action': 1, 'abnormal object': 2}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}

BATCH_SIZE   = 32
NUM_EPOCHS   = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-5
IMG_SIZE     = (224, 224)
RANDOM_SEED  = 42

Device: mps


## 3  Frame Extraction

Extract individual JPEG frames from every `.avi` test video.

In [3]:
def extract_frames(video_dir: Path, output_dir: Path, ext: str = '.jpg') -> None:
    """Extract all frames from every .avi file in *video_dir*."""
    output_dir.mkdir(parents=True, exist_ok=True)
    video_files = sorted(video_dir.glob('*.avi'))
    print(f'Found {len(video_files)} video(s).')

    for video_path in tqdm(video_files, desc='Extracting frames'):
        vid_name = video_path.stem
        vid_out  = output_dir / vid_name
        vid_out.mkdir(parents=True, exist_ok=True)

        cap = cv2.VideoCapture(str(video_path))
        for frame_idx in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):
            ret, frame = cap.read()
            if not ret:
                break
            cv2.imwrite(str(vid_out / f'{vid_name}_frame{frame_idx:04d}{ext}'), frame)
        cap.release()

    print('Done.')


extract_frames(VIDEO_DIR, FRAME_DIR)

Found 0 video(s).


Extracting frames: 0it [00:00, ?it/s]

Done.


## 4  Ground-Truth Loading

Each `vol<NN>.mat` file contains the frame indices where anomalies occur for one test video.

In [ ]:
def load_avenue_gt(gt_folder: Path) -> dict:
    """Return a dict mapping zero-padded video IDs to sets of anomalous frame numbers."""
    gt_map = {}
    for i in range(1, 22):
        vid      = f'{i:02d}'
        mat_path = gt_folder / f'vol{vid}.mat'
        if not mat_path.exists():
            print(f'Warning: {mat_path} not found.')
            gt_map[vid] = set()
            continue
        mat = scipy.io.loadmat(str(mat_path))
        for key in ('vol', 'gt', 'groundTruth'):
            if key in mat:
                gt_map[vid] = set(int(f) for f in mat[key].flatten())
                break
        else:
            print(f'Warning: no recognised GT key in {mat_path.name}')
            gt_map[vid] = set()
    return gt_map


gt_map = load_avenue_gt(GT_MAT_DIR)
print(f'Loaded GT for {len(gt_map)} videos — '
      f'{sum(len(v) for v in gt_map.values()):,} anomalous frames total.')

## 5  Rule-Based Label Generation

Priority order:
1. **abnormal object** — a bicycle or bag is detected by YOLOv8
2. **unusual action** — frame is in the GT anomaly set
3. **normal** — otherwise

In [ ]:
ABNORMAL_OBJECT_IDS = {1, 24, 26, 28}   # bicycle, backpack, handbag, suitcase


def detect_anomaly(yolo_labels: list, frame_num: int, video_id: str, gt_map: dict) -> str:
    """Assign an anomaly label to one frame via rule-based logic."""
    class_ids = set()
    for line in yolo_labels:
        line = line.strip()
        if line:
            try:
                class_ids.add(int(line.split()[0]))
            except (ValueError, IndexError):
                continue

    if class_ids & ABNORMAL_OBJECT_IDS:
        return 'abnormal object'
    if frame_num in gt_map.get(video_id, set()):
        return 'unusual action'
    return 'normal'


rows = []
for video_folder in tqdm(sorted(FRAME_DIR.iterdir()), desc='Labeling'):
    if not video_folder.is_dir():
        continue
    video_id = video_folder.name.zfill(2)
    for frame_file in sorted(video_folder.glob('*.jpg')):
        m = re.search(r'frame(\d+)', frame_file.name)
        if not m:
            continue
        frame_num  = int(m.group(1))
        label_file = LABEL_DIR / video_folder.name / frame_file.with_suffix('.txt').name
        yolo_labels = label_file.read_text().splitlines() if label_file.exists() else []
        label = detect_anomaly(yolo_labels, frame_num, video_id, gt_map)
        rows.append({'video': int(video_folder.name), 'frame': frame_num, 'anomaly_label': label})

df_labels = pd.DataFrame(rows)
df_labels.to_csv(CSV_PATH, index=False)
print(f'Saved {len(df_labels):,} labeled frames → {CSV_PATH}')
print(df_labels['anomaly_label'].value_counts())

## 6  Dataset

In [ ]:
class AvenueAnomalyDataset(Dataset):
    """
    PyTorch Dataset for Avenue frame-level anomaly classification.

    Args:
        dataframe:  DataFrame with columns 'video', 'frame', 'anomaly_label'.
        frame_root: Root directory containing per-video frame sub-folders.
        transform:  Optional torchvision transform.
    """

    def __init__(self, dataframe: pd.DataFrame, frame_root: Path, transform=None) -> None:
        self.data       = dataframe.reset_index(drop=True)
        self.frame_root = frame_root
        self.transform  = transform

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        row       = self.data.iloc[idx]
        vid       = str(int(row['video'])).zfill(2)
        frame_num = int(row['frame'])
        label     = LABEL2IDX.get(str(row['anomaly_label']), 0)
        path      = self.frame_root / vid / f'{vid}_frame{frame_num:04d}.jpg'
        image     = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

## 7  Model Architecture

ResNet-34 with a dropout + linear head. The attribute name `base` is preserved so the saved checkpoint is compatible with `src/models.py` used by `app.py`.

In [ ]:
class AnomalyClassifier(nn.Module):
    """ResNet-34 backbone with dropout + linear head for 3-class anomaly classification."""

    def __init__(self, num_classes: int = 3) -> None:
        super().__init__()
        # 'base' attribute name must match the state-dict keys expected by app.py.
        self.base = models.resnet34(weights='IMAGENET1K_V1')
        self.base.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.base.fc.in_features, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x)


# Sanity check
with torch.no_grad():
    _out = AnomalyClassifier()(torch.zeros(2, 3, 224, 224))
print(f'Output shape: {_out.shape}')   # expect (2, 3)

## 8  Training

- Grouped train/val split by video prevents data leakage
- LR scheduler steps **once per epoch**, not per batch
- Best checkpoint saved on highest val accuracy

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    # No ImageNet normalisation — kept consistent with app.py inference transform.
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

df = pd.read_csv(CSV_PATH)
df['label'] = df['anomaly_label'].map(LABEL2IDX)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, val_idx = next(splitter.split(df, groups=df['video']))
df_train, df_val = df.iloc[train_idx], df.iloc[val_idx]

print(f'Train: {len(df_train):,} | Val: {len(df_val):,}')
print(df_train['anomaly_label'].value_counts())

train_loader = DataLoader(
    AvenueAnomalyDataset(df_train, FRAME_DIR, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    AvenueAnomalyDataset(df_val, FRAME_DIR, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (model(images).argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100.0 * correct / total, all_labels, all_preds


model     = AnomalyClassifier(num_classes=3).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc   = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc, _, _ = evaluate(model, val_loader, DEVICE)
    scheduler.step()   # step once per epoch, not per batch

    for k, v in zip(
        ('train_loss', 'train_acc', 'val_loss', 'val_acc'),
        (tr_loss, tr_acc, vl_loss, vl_acc),
    ):
        history[k].append(v)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'final_op_avenue_model.pt')

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'Train loss {tr_loss:.4f}  acc {tr_acc:.1f}% | '
          f'Val   loss {vl_loss:.4f}  acc {vl_acc:.1f}%')

print(f'\nBest val accuracy: {best_val_acc:.1f}% — model saved to final_op_avenue_model.pt')

## 9  Evaluation

In [ ]:
# Load best checkpoint before final evaluation
model.load_state_dict(
    torch.load('final_op_avenue_model.pt', map_location=DEVICE, weights_only=True)
)

_, _, true_labels, pred_labels = evaluate(model, val_loader, DEVICE)
label_names = [IDX2LABEL[i] for i in range(3)]

print(classification_report(true_labels, pred_labels, target_names=label_names))

cm = confusion_matrix(true_labels, pred_labels)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Avenue Anomaly Classifier')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, NUM_EPOCHS + 1)
axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set(title='Loss', xlabel='Epoch')
axes[0].legend()
axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set(title='Accuracy (%)', xlabel='Epoch')
axes[1].legend()
plt.suptitle('Training History — Avenue Anomaly Classifier')
plt.tight_layout()
plt.show()

## 10  Inference Demo

Annotate a single video and write the result to disk.

In [ ]:
LABEL_COLORS_BGR = {
    'normal':          (50,  205,  50),
    'unusual action':  (0,   165, 255),
    'abnormal object': (0,     0, 220),
}

inference_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])


def annotate_video(input_path: str, output_path: str, model: nn.Module, device) -> None:
    """Write an annotated copy of *input_path* to *output_path*."""
    model.eval()
    cap    = cv2.VideoCapture(input_path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height)
    )

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        pil    = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        tensor = inference_transform(pil).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = F.softmax(model(tensor), dim=1).cpu().numpy().flatten()
        pred_idx   = int(probs.argmax())
        confidence = float(probs[pred_idx])
        label      = IDX2LABEL[pred_idx]
        cv2.putText(
            frame, f'{label} ({confidence:.2f})',
            (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
            LABEL_COLORS_BGR[label], 2, cv2.LINE_AA,
        )
        writer.write(frame)

    cap.release()
    writer.release()
    print(f'Saved → {output_path}')


annotate_video(
    str(VIDEO_DIR / '01.avi'),
    'avenue_annotated_01.mp4',
    model, DEVICE,
)